# Facial Emotion Detection

This notebook demonstrates how to detect emotions from faces in images using OpenCV and computer vision techniques.

**Note:** This implementation uses a simple heuristic for demonstration purposes. For accurate emotion detection, you should use a trained deep learning model such as:
- FER-2013 dataset trained CNN models
- Pre-trained models from TensorFlow/Keras
- Custom trained emotion classification models

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import os

# Emotion labels (FER-2013 dataset emotions)
EMOTIONS = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

In [ ]:
# Load face detection cascade
face_cascade_path = "../models/haarcascade_frontalface_default.xml"
face_cascade = cv2.CascadeClassifier(face_cascade_path)

if face_cascade.empty():
    print("Error: Could not load face cascade classifier")
else:
    print("Face cascade loaded successfully")

In [ ]:
def predict_emotion_simple(face_roi):
    """
    Simple emotion prediction based on basic image features.
    This is a placeholder - replace with actual trained model.

    Args:
        face_roi: Normalized face region (48x48)

    Returns:
        tuple: (predicted_emotion, confidence)
    """
    # Calculate some basic features
    mean_intensity = np.mean(face_roi)
    std_intensity = np.std(face_roi)

    # Simple rules (for demonstration - not accurate)
    if std_intensity > 0.15:  # High contrast might indicate surprise or anger
        if mean_intensity > 0.6:  # Bright face
            return 'Surprise', 0.75
        else:  # Dark face
            return 'Angry', 0.70
    elif mean_intensity > 0.7:  # Very bright face
        return 'Happy', 0.80
    elif mean_intensity < 0.3:  # Dark face
        return 'Sad', 0.65
    else:
        return 'Neutral', 0.60

def detect_faces_and_emotions(image):
    """
    Detect faces and their emotions in an image.

    Args:
        image: Input image (RGB)

    Returns:
        tuple: (image_with_detections, results)
    """
    # Convert to grayscale for face detection
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    # Detect faces
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(30, 30)
    )

    image_with_detections = image.copy()
    results = []

    for (x, y, w, h) in faces:
        # Extract face ROI
        face_roi = gray[y:y+h, x:x+w]

        # Resize to standard size for emotion detection
        face_roi_resized = cv2.resize(face_roi, (48, 48))

        # Normalize
        face_roi_normalized = face_roi_resized / 255.0

        # Predict emotion
        emotion, confidence = predict_emotion_simple(face_roi_normalized)

        # Draw rectangle around face
        cv2.rectangle(image_with_detections, (x, y), (x+w, y+h), (0, 255, 0), 2)

        # Draw emotion label
        label = f"{emotion}: {confidence:.2f}"
        cv2.putText(image_with_detections, label, (x, y-10),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

        results.append({
            'emotion': emotion,
            'confidence': confidence,
            'bbox': (x, y, w, h)
        })

    return image_with_detections, results

In [ ]:
# Load a test image
image_path = "../data/test_images/car.jpg"  # You might want to use an image with faces
if os.path.exists(image_path):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    plt.imshow(image)
    plt.title("Original Image")
    plt.show()
else:
    print(f"Test image not found at {image_path}")
    # Create a dummy image for demonstration
    image = np.random.randint(0, 255, (300, 400, 3), dtype=np.uint8)
    plt.imshow(image)
    plt.title("Dummy Test Image")
    plt.show()

In [ ]:
# Test emotion detection
result_image, emotion_results = detect_faces_and_emotions(image)

# Display results
plt.figure(figsize=(12, 8))

plt.subplot(1, 2, 1)
plt.imshow(image)
plt.title("Original Image")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(result_image)
plt.title("Emotion Detection Results")
plt.axis('off')

plt.show()

# Print detected emotions
print("Detected Emotions:")
for i, result in enumerate(emotion_results):
    print(f"Face {i+1}: {result['emotion']} (Confidence: {result['confidence']:.2f})")
    print(f"  Bounding Box: {result['bbox']}")

if not emotion_results:
    print("No faces detected in the image.")